In [12]:
import pandas as pd
import glob

# 日度表
daily_files = glob.glob('data/TRD_Dalyr*.xlsx')
daily = pd.concat([pd.read_excel(f, header=0, skiprows=[1,2]) for f in daily_files])

# 月度表
monthly = pd.read_excel('data/TRD_Mnth.xlsx', header=0, skiprows=[1,2])

daily.to_pickle('daily.pkl')
monthly.to_pickle('monthly.pkl')

print(daily.shape)
print(monthly.shape)
print(daily.head())
print(monthly.head())

d:\Anaconda3\envs\dataanalysis\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\Anaconda3\envs\dataanalysis\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\Anaconda3\envs\dataanalysis\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\Anaconda3\envs\dataanalysis\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\Anaconda3\envs\dataanalysis\Lib\site-packages\openpyxl\styles\stylesheet.py:237: 

(13840496, 7)
(691885, 6)
   Stkcd       Trddt  Clsprc  Dnshrtrd      Dsmvosd      Dsmvtll    Dretwd
0      1  2011-09-15   16.56  17064236  51424739.61  84842682.89  0.008526
1      1  2011-09-16   17.00  36953539  52791097.42  87096957.07  0.026570
2      1  2011-09-19   16.60  10599840  51548953.96  85047616.91 -0.023529
3      1  2011-09-20   16.74   7712700  51983704.17  85764885.96  0.008434
4      1  2011-09-21   17.06  18368645  52977418.94  87404358.10  0.019116
   Stkcd   Trdmnt  Mclsprc      Msmvosd      Msmvttl    Mretwd
0      1  2011-09    16.04  49809953.10  82178540.67 -0.048071
1      1  2011-10    16.88  52418454.38  86482155.02  0.052369
2      1  2011-11    15.51  48164113.00  79463164.95 -0.081161
3      1  2011-12    15.59  48412541.70  79873032.99  0.005158
4      1  2012-01    16.64  51673168.30  85252550.92  0.067351


In [31]:
daily['turnover'] = daily['Dnshrtrd'] / (daily['Dsmvosd'] * 1000 / daily['Clsprc'])
daily['ym'] = pd.to_datetime(daily['Trddt']).dt.to_period('M')
monthly_turnover = daily.groupby(['Stkcd', 'ym'])['turnover'].mean().reset_index()

() 元组（tuple）：不可修改的有序序列，比如 (1, 2, 3)，创建后不能增删改

[] 列表（list）：可修改的有序序列，比如 [1, 2, 3]，可以append、删除、改值

{} 字典（dict）：键值对，比如 {'name': 'Haco', 'age': 24}，用key查value

DataFrame是pandas自己定义的数据结构，本质上是一个二维表。

python
{
  'Stkcd':    ['000001', '000001', '000002', ...],
  'Trddt':    ['2011-01-04', '2011-01-05', ...],
  'Clsprc':   [15.20, 15.35, ...],
  'Dnshrtrd': [5000000, 3200000, ...]
}

pandas把这个包装成DataFrame，加上了行索引、列名、各种方法（筛选、排序、groupby等），让你操作表格方便。但底层存数据就是按列存的。

In [32]:
monthly['ym'] = pd.to_datetime(monthly['Trdmnt']).dt.to_period('M')
df = pd.merge(monthly_turnover, monthly, on=['Stkcd', 'ym'])
print(df.shape)
print(df.head())

(691878, 8)
   Stkcd       ym  turnover   Trdmnt  Mclsprc      Msmvosd      Msmvttl  \
0      1  2011-09  0.005107  2011-09    16.04  49809953.10  82178540.67   
1      1  2011-10  0.005553  2011-10    16.88  52418454.38  86482155.02   
2      1  2011-11  0.004961  2011-11    15.51  48164113.00  79463164.95   
3      1  2011-12  0.004808  2011-12    15.59  48412541.70  79873032.99   
4      1  2012-01  0.008461  2012-01    16.64  51673168.30  85252550.92   

     Mretwd  
0 -0.048071  
1  0.052369  
2 -0.081161  
3  0.005158  
4  0.067351  


In [33]:
df.to_pickle('merged.pkl')

In [34]:
print(daily.columns.tolist())

['Stkcd', 'Trddt', 'Clsprc', 'Dnshrtrd', 'Dsmvosd', 'Dsmvtll', 'Dretwd', 'turnover', 'ym']


In [35]:
# 每只股票最早出现的月份
first_month = df.groupby('Stkcd')['ym'].min().reset_index()
first_month.columns = ['Stkcd', 'first_ym']
df = pd.merge(df, first_month, on='Stkcd')

# 计算距首次出现的月数，剔除不满6个月的
df['months_since_ipo'] = (df['ym'] - df['first_ym']).apply(lambda x: x.n)
df = df[df['months_since_ipo'] >= 6]

print(df.shape)

(656685, 10)


In [36]:
from scipy.stats.mstats import winsorize

df['turnover_win'] = winsorize(df['turnover'], limits=[0.01, 0.01])
print(df['turnover_win'].describe())

count    656685.000000
mean          0.027479
std           0.028976
min           0.001063
25%           0.008883
50%           0.017543
75%           0.034759
max           0.158886
Name: turnover_win, dtype: float64


d:\Anaconda3\envs\dataanalysis\Lib\site-packages\numpy\lib\_function_base_impl.py:4842: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


In [37]:
df.to_pickle('cleaned.pkl')

# 每月按换手率分5组
df['quintile'] = df.groupby('ym')['turnover_win'].transform(
    lambda x: pd.qcut(x, 5, labels=[1,2,3,4,5])
)

# 每组每月的等权平均收益
group_ret = df.groupby(['ym', 'quintile'])['Mretwd'].mean().reset_index()

# 看每组的平均月收益
print(group_ret.groupby('quintile')['Mretwd'].mean())

quintile
1   -0.005644
2   -0.003427
3    0.002745
4    0.013751
5    0.053768
Name: Mretwd, dtype: float64


C:\Users\10914\AppData\Local\Temp\ipykernel_31556\220863544.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_ret = df.groupby(['ym', 'quintile'])['Mretwd'].mean().reset_index()
C:\Users\10914\AppData\Local\Temp\ipykernel_31556\220863544.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(group_ret.groupby('quintile')['Mretwd'].mean())


In [38]:
# 按股票排序，生成下月收益
df = df.sort_values(['Stkcd', 'ym'])
df['next_ret'] = df.groupby('Stkcd')['Mretwd'].shift(-1)

# 重新分组，用下月收益
group_ret = df.groupby(['ym', 'quintile'])['next_ret'].mean().reset_index()
print(group_ret.groupby('quintile')['next_ret'].mean())

quintile
1    0.013435
2    0.015621
3    0.015441
4    0.014471
5    0.005205
Name: next_ret, dtype: float64


C:\Users\10914\AppData\Local\Temp\ipykernel_31556\145643623.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  group_ret = df.groupby(['ym', 'quintile'])['next_ret'].mean().reset_index()
C:\Users\10914\AppData\Local\Temp\ipykernel_31556\145643623.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(group_ret.groupby('quintile')['next_ret'].mean())


In [39]:
# 每月Q1减Q5的收益
long_short = group_ret.pivot(index='ym', columns='quintile', values='next_ret')
long_short['LS'] = long_short[1] - long_short[5]

print('多空月均收益:', long_short['LS'].mean())
print('标准差:', long_short['LS'].std())
print('t统计量:', long_short['LS'].mean() / long_short['LS'].std() * (len(long_short)**0.5))

多空月均收益: 0.008229250280203618
标准差: 0.060069854705793756
t统计量: 1.8070838873327122


In [40]:
from scipy.stats import spearmanr

# 每月算换手率和下月收益的Spearman相关系数
ic_list = []
for ym, group in df.dropna(subset=['next_ret']).groupby('ym'):
    corr, _ = spearmanr(group['turnover_win'], group['next_ret'])
    ic_list.append({'ym': ym, 'IC': corr})

ic_df = pd.DataFrame(ic_list)
print('IC均值:', ic_df['IC'].mean())
print('IC标准差:', ic_df['IC'].std())
print('ICIR:', ic_df['IC'].mean() / ic_df['IC'].std())

IC均值: -0.07317772829340095
IC标准差: 0.18356028889623985
ICIR: -0.3986577311107074


In [41]:
import statsmodels.api as sm

# 读FF3数据
ff3 = pd.read_excel('data/STK_MKT_THRFACMONTH.xlsx', header=0, skiprows=[1,2])
ff3 = ff3[ff3['MarkettypeID'] == 'P9714']
ff3['ym'] = pd.to_datetime(ff3['TradingMonth']).dt.to_period('M')

# 合并多空收益和FF3因子
reg_df = pd.merge(long_short[['LS']].reset_index(), ff3[['ym', 'RiskPremium1', 'SMB1', 'HML1']], on='ym')
reg_df = reg_df.dropna()

# OLS回归
X = sm.add_constant(reg_df[['RiskPremium1', 'SMB1', 'HML1']])
y = reg_df['LS']
result = sm.OLS(y, X).fit()
print(result.summary())

                            OLS Regression Results                            
Dep. Variable:                     LS   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                 -0.004
Method:                 Least Squares   F-statistic:                    0.7762
Date:                Tue, 15 Sep 2026   Prob (F-statistic):              0.509
Time:                        18:51:03   Log-Likelihood:                 242.73
No. Observations:                 173   AIC:                            -477.5
Df Residuals:                     169   BIC:                            -464.8
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0068      0.005      1.452   

d:\Anaconda3\envs\dataanalysis\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [42]:
df.to_pickle('final.pkl')
long_short.to_pickle('long_short.pkl')
ic_df.to_pickle('ic_df.pkl')